# 07. Bayesian hyperparameter optimisation of LightGBM (Optuna TPE)
**Run on:** Laptop B · **Approx. runtime:** ~10-12 min on a 4-core laptop

LightGBM was the best model in the comparison, so it is tuned here.
- **Search:** Optuna's Tree-structured Parzen Estimator (TPE, a Bayesian method), seed 42, 8 random start-up trials, then model-guided.
- **Objective:** maximise the mean **macro F1** over **3-fold GroupKFold by trip** on the training trips (150k-row samples per fold for speed).
- **Afterwards:** the untuned and tuned models are compared with the **same** 5-fold CV and student-test protocol.

**Before running:** the dataset must be in `data/train/<id>/` and `data/student_test/<id>/`, and `pip install -r requirements.txt` done. Run all cells top to bottom (Kernel → Restart & Run All). Every number and plot below is computed when you run it.

In [ ]:
import os, sys, warnings
# run from notebooks/ or from the repo root
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
warnings.filterwarnings("ignore")
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160)
from pathsnap import nb
from pathsnap.models import make_model
from pathsnap.config import CFG
SEED = CFG["seed"]  # 42: every model and data split uses this seed
print("repo root:", ROOT)

In [ ]:
tr, tr_pts, te, te_pts = nb.load_data()

## Search space

In [ ]:
import optuna, time
optuna.logging.set_verbosity(optuna.logging.WARNING)
sys.path.insert(0, os.path.join(ROOT, "scripts"))
from importlib import import_module
space = import_module("03_bayes_opt").space   # the same search space as scripts/03_bayes_opt.py
import inspect; print(inspect.getsource(space).split('if name == "rf"')[0])

In [ ]:
N_TRIALS = 20     # increase for a longer search
from pathsnap.experiment import cross_validate

def objective(trial):
    params = space(trial, "lightgbm")
    folds = cross_validate(lambda: make_model("lightgbm", SEED, **params), tr, tr_pts, folds=3,
                           max_rows=150_000, seed=SEED, verbose=False)
    f1 = float(np.mean([f["macro_f1"] for f in folds]))
    print(f"trial {trial.number:2d}: CV macro F1 {f1:.4f}")
    return f1

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=8))
t0 = time.time(); study.optimize(objective, n_trials=N_TRIALS); search_s = time.time() - t0
print(f"\nsearch time {search_s:.0f}s | best CV macro F1 {study.best_value:.4f}")
study.best_params

## Search record and plots

In [ ]:
trials = study.trials_dataframe()
os.makedirs("results/notebooks", exist_ok=True); trials.to_csv("results/notebooks/optuna_trials_lightgbm.csv", index=False)
display(trials[["number", "value"] + [c for c in trials if c.startswith("params_")]].round(4))
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(trials.number, trials.value, "o", label="trial"); ax.plot(trials.number, trials.value.cummax(), "r-", label="best so far")
ax.set(xlabel="trial", ylabel="3-fold CV macro F1", title="Optimisation history"); ax.legend(); nb.show()

In [ ]:
imp = optuna.importance.get_param_importances(study)
plt.figure(figsize=(6, 3.5)); plt.barh(list(imp)[::-1], list(imp.values())[::-1])
plt.title("Hyperparameter importance (fANOVA)"); plt.xlabel("importance"); nb.show()
ps = [c for c in trials if c.startswith("params_")]
fig, axs = plt.subplots(2, 4, figsize=(15, 6))
for a, c in zip(axs.ravel(), ps):
    a.scatter(trials[c], trials.value, c=trials.number, cmap="viridis"); a.set(title=c[7:], ylabel="CV macro F1")
    if c[7:] in ("learning_rate", "num_leaves", "min_child_samples", "reg_lambda"): a.set_xscale("log")
axs.ravel()[-1].axis("off"); fig.suptitle("Slice plots (colour = trial number)"); fig.tight_layout(); nb.show()

## Untuned vs tuned (same 5-fold CV and student-test protocol)

In [ ]:
res = {}
for label, params in [("untuned", {}), ("tuned", study.best_params)]:
    print(f"=== {label}")
    make = lambda p=params: make_model("lightgbm", SEED, **p)
    cv = nb.run_cv(make, tr, tr_pts)
    model, train_m, test_m, p = nb.fit_and_evaluate(make, tr, tr_pts, te, te_pts)
    res[label] = (cv, train_m, test_m, model)
table = pd.DataFrame({k: {"CV macro F1": v[0].macro_f1.mean(), "CV std": v[0].macro_f1.std(), "test Top-1": v[2]["top1"],
                          "test macro F1": v[2]["macro_f1"], "test log loss": v[2]["log_loss"],
                          "test continuity": v[2]["continuity"], "inference ms/point": v[2]["infer_ms_per_point"],
                          "fit seconds": v[2]["fit_s"]} for k, v in res.items()}).round(4)
table

In [ ]:
ms = ["CV macro F1", "test Top-1", "test macro F1", "test log loss"]
x = np.arange(len(ms)); plt.figure(figsize=(8, 3.8))
plt.bar(x - .2, table.loc[ms, "untuned"], .4, label="untuned"); plt.bar(x + .2, table.loc[ms, "tuned"], .4, label="tuned")
plt.xticks(x, ms); plt.title("Untuned vs Bayesian-tuned LightGBM"); plt.legend(); nb.show()
d = table.loc["CV macro F1", "tuned"] - table.loc["CV macro F1", "untuned"]
print(f"CV macro F1 change: {d:+.4f} (fold std about {table.loc['CV std', 'untuned']:.4f})")

In [ ]:
import joblib
joblib.dump(res["tuned"][3], "results/notebooks/lightgbm_tuned_notebook.joblib")
print("tuned model saved to results/notebooks/ (the deployed model stays artifacts/models/final.joblib)")